## Trabajo práctico 3

### Autor: Emmanuel Guerreiro - 47262

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin


In [ ]:
TARGET_COLUMN = 'is_increment_24hs'

In [3]:
import pandas as pd

raw_df = pd.read_csv('data.csv')
raw_df.head()

,date,text,favorites,retweets,Toxic,Insult,Profanity,Derogatory,Sexual,"Death, Harm & Tragedy",...,Illicit Drugs,War & Conflict,Politics,Finance,Legal,btc_tweet_day,btc_24h_after,btc_48h_after,btc_delta_24h,btc_delta_48h
0,2020-03-03 01:34:00,I was thrilled to be back in the Great city of...,73748,17404,0.014622,0.010394,0.002904,0.003574,0.002435,0.059701,...,0.036585,0.472222,0.867769,0.079710,0.435185,8753.01,8700.00,9085.48,-53.01,332.47
1,2020-01-17 03:22:00,RT @CBS_Herridge: READ: Letter to surveillance...,0,7396,0.050346,0.050961,0.023985,0.008934,0.010127,0.085399,...,0.294118,0.350000,0.680934,0.081967,0.948357,8850.83,8903.26,8631.95,52.43,-218.88
2,2020-09-12 20:10:00,The Unsolicited Mail In Ballot Scam is a major...,80527,23502,0.258527,0.112138,0.058611,0.043741,0.015008,0.082609,...,0.700000,0.058824,0.937500,0.515901,0.858108,10359.99,10280.14,10698.03,-79.85,338.04
3,2020-01-17 13:13:00,RT @MZHemingway: Very friendly telling of even...,0,9081,0.018771,0.012035,0.004276,0.003942,0.002470,0.207207,...,0.057377,0.375000,0.912500,0.079710,0.333333,8850.83,8903.26,8631.95,52.43,-218.88
4,2020-01-17 00:11:00,RT @WhiteHouse: President @realDonaldTrump ann...,0,25048,0.016713,0.011194,0.004276,0.005187,0.002435,0.073482,...,0.057377,0.141892,0.969512,0.102302,0.948357,8850.83,8903.26,8631.95,52.43,-218.88


In [4]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop)

In [ ]:
class GenerateTarget(BaseEstimator, TransformerMixin):
    """
    Transformer that generates a binary target column (`target_movement`)
    from the numeric feature `btc_delta_24h_pct`.
    
    Returns the same DataFrame with the new column appended.
    """

    def fit(self, X, y=None):
        
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        
        X = X.copy()

        X[TARGET_COLUMN] = (X['btc_delta_24h'] > 0).astype(int)

        return X


In [6]:
import re
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class NormalizeData(BaseEstimator, TransformerMixin):
    """Transformer that normalizes column names to snake_case."""

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    @staticmethod
    def to_snake_case(name):
        """Convert a string to snake_case."""
        s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
        s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
        s3 = re.sub(r'[-\s]+', '_', s2)
        s4 = re.sub(r'_+', '_', s3)
        return s4.strip('_').lower()

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [self.to_snake_case(col) for col in df.columns]
        return df


In [ ]:
import numpy as np

class ManualFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self
        
    def transform(self, df):
        
        df['btc_delta_24h_pct'] = df['btc_delta_24h'] / df['btc_tweet_day']
        df['btc_delta_48h_pct'] = df['btc_delta_48h'] / df['btc_tweet_day']

        # Hora del tweet

        # Feature que junta otras de google

        # Otra feature mas

        return df

In [ ]:
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, FeatureUnion

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Define preprocessors
column_transformer = ColumnTransformer([
    ('num', numeric_pipeline,  make_column_selector(dtype_include=["int64", "float64"])),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=["object", "category"]))
])

features_pipelines = FeatureUnion([
    ("feature_engineer", ManualFeatureEngineering()),  # keep all original features
    ('pca', PCA(n_components='mle'))         
])

preprocessor = Pipeline([
    ("normalize_data", NormalizeData()),
    ("features_pipelines", features_pipelines),
    # ("drop_columns", DropColumns(columns_to_drop=[])),
    ("column_transformer", column_transformer),
])

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

data_prep_pipeline = Pipeline([
    ("generate_target", GenerateTarget()),
])

df_prepared = data_prep_pipeline.fit_transform(raw_df)

y = df_prepared[TARGET_COLUMN]
X = df_prepared.drop(columns=[TARGET_COLUMN])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [10]:
from sklearn.model_selection import KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)


In [12]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

# Pipeline
rf_cv_pca_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier
    (
        class_weight='balanced',
        random_state=42
    ))
])

# Grilla de parámetros
param_grid_rf = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    rf_cv_pca_pipeline,
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

ValueError: 
All the 135 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "/home/emma/facultad/ds/repositorio/integrador/tercer-entrega/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/emma/facultad/ds/repositorio/integrador/tercer-entrega/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/emma/facultad/ds/repositorio/integrador/tercer-entrega/.venv/lib/python3.12/site-packages/sklearn/pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_para

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

print("Best parameters (Random Forest):")
print(grid_rf.best_params_)

def evaluar_modelo(nombre, y_true, y_pred):
    print(f"\n🔎 {nombre}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=3))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

y_pred_logreg_grid = grid_rf.predict(X_test)

evaluar_modelo("Random Forest", y_test, y_pred_logreg_grid)

Best parameters (Random Forest):
{'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 50}

🔎 Random Forest

Classification report:
              precision    recall  f1-score   support

           0      0.471     0.461     0.466      2089
           1      0.540     0.550     0.545      2402

    accuracy                          0.509      4491
   macro avg      0.505     0.505     0.505      4491
weighted avg      0.508     0.509     0.508      4491

Confusion matrix:
[[ 962 1127]
 [1080 1322]]


## XGBoost 

In [ ]:
### Class weights to make the model more balanced

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute weights proportional to inverse frequency
classes = np.unique(y_train_encoded)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_encoded
)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

In [ ]:
from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42, k_neighbors=5)
smote = SMOTE(sampling_strategy={3: 500}, random_state=42)



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline


xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

pipeline_xgb_smote = ImbPipeline([
    ("feature_engineer", ManualFeatureEngineering()),
    ("drop_columns", DropColumns(columns_to_drop=['wage', 'value', 'gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning', 'gk_reflexes', 'club_rating'])),
    ("column_transformer", column_transformer),
    # ("smote", smote),
    ("model", xgb_model)
])
param_grid_xgb = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

grid_xgb = GridSearchCV(
    pipeline_xgb_smote,
    param_grid=param_grid_xgb,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

# grid_xgb.fit(X_train, y_train_encoded, model__sample_weight=np.array([class_weights[label] for label in y_train_encoded]))
grid_xgb.fit(X_train, y_train_encoded)

y_pred_xgb_grid = grid_xgb.predict(X_test)
evaluar_modelo("XGBoost", y_test_encoded, y_pred_xgb_grid)

In [ ]:
test_df = pd.read_csv('test.csv')


# ✅ 2. Get predictions from your trained GridSearchCV
best_model = grid_xgb.best_estimator_
y_pred = best_model.predict(test_df)

# ✅ 3. Define your class mapping (as in your training)
class_mapping = {
    'Arquero': np.int64(0),
    'Defensor central': np.int64(1),
    'Delantero': np.int64(2),
    'Extremo': np.int64(3),
    'Lateral': np.int64(4),
    'Volante': np.int64(5),
    'Volante defensivo': np.int64(6),
}

# ✅ 4. Invert the mapping to go from numeric → string label
inverse_mapping = {v: k for k, v in class_mapping.items()}

# ✅ 5. Map numeric predictions to their labels
y_pred_labels = [inverse_mapping[int(label)] for label in y_pred]

# ✅ 6. Build the submission DataFrame
submission = pd.DataFrame({
    "id": test_df["id"],
    "posicion": y_pred_labels
})

name = f"submission.csv"

# ✅ 7. Save to CSV
submission.to_csv(name, index=False)

print("✅ submission.csv generated successfully!")
print(submission.head())